# 03 · Subdivision / Region Analysis

One row per country × state/province per month.

"Subdivision1" is the first administrative level below country — state in the US,
Land in Germany, province in Canada. Region names come from the **MaxMind GeoIP2**
database and are in English.

> **Schema caveat** — The subdivision column name can differ between the download
> and upload parquet files (e.g. `subdivision1` vs. a different name). This notebook
> detects the column name at runtime and normalises it before merging, making it
> robust to schema drift across M-Lab releases.

## Setup

In [1]:
import json
from pathlib import Path

import pandas as pd
import requests

try:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
    import seaborn as sns
    from IPython.display import clear_output, display
    sns.set_theme(style="whitegrid", palette="muted")
    plt.rcParams["figure.figsize"] = (12, 5)
except ImportError as e:
    print(f"Note: {e}")
    print("  Install with: uv add matplotlib seaborn ipywidgets")


In [2]:
# ── Country name lookup ──────────────────────────────────────────────────────
# countrylookup.py is a local helper (same directory as this notebook) that
# converts ISO 3166-1 alpha-2 codes to readable English country names.
# It tries pycountry → restcountries.com API → built-in fallback dict.
#
# If you move this notebook, keep countrylookup.py alongside it.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # ensure local module is found
from countrylookup import cc_name, cc_label

print(f"Country lookup ready — {cc_label('US')}, {cc_label('KR')}, {cc_label('XK')}")

[countrylookup] downloading country names from restcountries.com ...
Country lookup ready — United States (US), South Korea (KR), Kosovo (XK)


In [3]:
# ── Discover available months ─────────────────────────────────────────────────
#
# Fetch the M-Lab manifest to learn which months are available per slice.
# Each entry includes the public download URL and the local cache path.

MANIFEST_URL = "https://measurementlab.net/data/iqb/manifest.json"
resp = requests.get(MANIFEST_URL, timeout=30)
resp.raise_for_status()

records = []
for path, meta in resp.json()["files"].items():
    parts = path.split("/")
    if len(parts) >= 6 and parts[5] == "data.parquet":
        records.append({
            "start":      pd.to_datetime(parts[2], format="%Y%m%dT%H%M%SZ"),
            "slice":      parts[4],
            "url":        meta["url"],
            "cache_path": path,
        })

catalog = (
    pd.DataFrame(records)
    .sort_values(["slice", "start"])
    .reset_index(drop=True)
)

print(f"Catalog: {len(catalog)} entries, {catalog['slice'].nunique()} slices, "
      f"{catalog['start'].min().date()} → {catalog['start'].max().date()}")


Catalog: 2450 entries, 12 slices, 2009-01-01 → 2026-01-01


In [4]:
# ── Data loader ──────────────────────────────────────────────────────────────
#
# Downloads parquet files from the public M-Lab URLs in the manifest.
# Files are cached to ./cache/v1/... on first access and reused on subsequent
# runs (matching the path structure used by the iqb library's local cache).

from io import BytesIO

_mem_cache: dict = {}

def load_parquet(slice_name: str, start: str) -> pd.DataFrame:
    key = (slice_name, start)
    if key in _mem_cache:
        return _mem_cache[key]

    start_ts = pd.to_datetime(start)
    row = catalog[(catalog["slice"] == slice_name) & (catalog["start"] == start_ts)]
    if row.empty:
        available = catalog[catalog["slice"] == slice_name]["start"].dt.strftime("%Y-%m-%d").tolist()
        raise ValueError(f"No data for slice='{slice_name}', month='{start}'.\nAvailable: {available}")
    row = row.iloc[0]

    local_path = Path(row["cache_path"])
    if local_path.exists():
        df = pd.read_parquet(local_path)
    else:
        print(f"[download] {slice_name} / {start} …")
        r = requests.get(row["url"], timeout=60)
        r.raise_for_status()
        local_path.parent.mkdir(parents=True, exist_ok=True)
        local_path.write_bytes(r.content)
        df = pd.read_parquet(BytesIO(r.content))
        print(f"  ✓ saved to {local_path}  ({len(df):,} rows)")

    _mem_cache[key] = df
    return df


## Interactive Subdivision Explorer

Select a month and country to rank all subdivisions by your chosen metric.

In [5]:
sub_months = sorted(
    catalog[catalog["slice"] == "downloads_by_country_subdivision1"]["start"]
    .dt.strftime("%Y-%m-%d").unique(), reverse=True,
)

# Metric options: display label → (column name, lower_is_better)
# Latency and loss: lower raw values are better, but IQB's polarity inversion
# means higher percentiles represent the better-performing connections.
METRICS = {
    "Download p50 (Mbit/s)":       ("download_p50",  False),
    "Upload p50 (Mbit/s)":         ("upload_p50",    False),
    "Latency p50 (ms)":            ("latency_p50",   True),
    "Packet Loss p50 (fraction)":  ("loss_p50",      True),
}

w_month   = widgets.Dropdown(options=sub_months, description="Month:",
                              layout=widgets.Layout(width="250px"))
w_country = widgets.Dropdown(options=[],         description="Country:",
                              layout=widgets.Layout(width="200px"))
w_topn    = widgets.IntSlider(value=20, min=5, max=70, step=5,
                               description="Top N:", layout=widgets.Layout(width="380px"))
w_metric  = widgets.Dropdown(options=list(METRICS), description="Metric:",
                              layout=widgets.Layout(width="290px"))
out       = widgets.Output()

_sub: dict = {}
def _load_sub(month):
    if month in _sub: return _sub[month]
    dl = load_parquet("downloads_by_country_subdivision1", month)
    ul = load_parquet("uploads_by_country_subdivision1",   month)
    # Detect subdivision column — name may differ between download and upload frames
    dl_col = next((c for c in dl.columns if "subdivision" in c.lower()), "subdivision1")
    ul_col = next((c for c in ul.columns if "subdivision" in c.lower()), dl_col)
    if ul_col != dl_col:
        ul = ul.rename(columns={ul_col: dl_col})
    _sub[month] = (dl, ul, dl_col)
    return _sub[month]

def on_month(change):
    prev_country = w_country.value
    dl, _, _ = _load_sub(change["new"])
    cc = sorted(dl["country_code"].dropna().unique())
    w_country.options = [(cc_label(c), c) for c in cc]
    w_country.value = prev_country if prev_country in cc else ("US" if "US" in cc else cc[0])
    draw()

def draw(change=None):
    month, country = w_month.value, w_country.value
    if not country: return
    col, lower = METRICS[w_metric.value]
    dl, ul, sc = _load_sub(month)
    df = dl[dl["country_code"]==country].merge(
         ul[ul["country_code"]==country][[sc,"upload_p50"]], on=sc, how="left"
         ).dropna(subset=[sc])
    top = df.nsmallest(w_topn.value, col) if lower else df.nlargest(w_topn.value, col)
    top = top.sort_values(col, ascending=not lower)
    with out:
        clear_output(wait=True)
        if top.empty: print(f"No subdivision data for {country} / {month}."); return
        fig, ax = plt.subplots(figsize=(11, max(5, w_topn.value*0.38)))
        ax.barh(top[sc], top[col])
        ax.set_xlabel(w_metric.value)
        ax.set_title(f"Top {w_topn.value} subdivisions — {cc_name(country)} — "
                     f"{w_metric.value} {'(lower=better)' if lower else ''}\n{month}")
        plt.tight_layout(); plt.show()

w_month.observe(on_month,"value"); w_country.observe(draw,"value")
w_topn.observe(draw,"value"); w_metric.observe(draw,"value")
display(widgets.VBox([widgets.HBox([w_month,w_country,w_metric]), w_topn, out]))
on_month({"new": sub_months[0]})

[download] downloads_by_country_subdivision1 / 2025-12-01 …
  ✓ saved to cache/v1/20251201T000000Z/20260101T000000Z/downloads_by_country_subdivision1/data.parquet  (2,448 rows)
[download] uploads_by_country_subdivision1 / 2025-12-01 …
  ✓ saved to cache/v1/20251201T000000Z/20260101T000000Z/uploads_by_country_subdivision1/data.parquet  (2,425 rows)


## Download vs Upload by Region

Comparing download and upload medians within a country reveals infrastructure
differences by region. Areas served primarily by cable tend to be highly asymmetric
(fast download, slow upload). Fibre-heavy areas appear close to the diagonal.

In [6]:
w_sm = widgets.Dropdown(options=sub_months, description="Month:",
                         layout=widgets.Layout(width="250px"))
w_sc = widgets.Dropdown(options=[],         description="Country:",
                         layout=widgets.Layout(width="200px"))
out_s = widgets.Output()

def on_sm(change):
    prev = w_sc.value
    dl,_,_ = _load_sub(change["new"])
    cc = sorted(dl["country_code"].dropna().unique())
    w_sc.options = [(cc_label(c), c) for c in cc]
    w_sc.value = prev if prev in cc else ("US" if "US" in cc else cc[0])
    draw_s()

def draw_s(change=None):
    dl, ul, sc = _load_sub(w_sm.value)
    country = w_sc.value
    if not country: return
    df = dl[dl["country_code"]==country].merge(
         ul[ul["country_code"]==country][[sc,"upload_p50"]], on=sc, how="inner"
         ).dropna(subset=[sc])
    with out_s:
        clear_output(wait=True)
        if df.empty: print(f"No data for {country}."); return
        fig, ax = plt.subplots(figsize=(8,8))
        ax.scatter(df["download_p50"],df["upload_p50"],alpha=0.65,
                   edgecolors="white",linewidths=0.5)
        lim = max(df["download_p50"].max(),df["upload_p50"].max())*1.05
        ax.plot([0,lim],[0,lim],"k--",lw=0.8,alpha=0.4)
        for _, row in df.iterrows():
            ax.annotate(row[sc],(row["download_p50"],row["upload_p50"]),
                        fontsize=7,alpha=0.65)
        ax.set_xlabel("Median download (Mbit/s)"); ax.set_ylabel("Median upload (Mbit/s)")
        ax.set_title(f"Download vs upload by subdivision — {cc_name(country)} — {w_sm.value}")
        plt.tight_layout(); plt.show()

w_sm.observe(on_sm,"value"); w_sc.observe(draw_s,"value")
display(widgets.VBox([widgets.HBox([w_sm,w_sc]),out_s]))
on_sm({"new": sub_months[0]})